In [1]:
import torch 
import os
import numpy as np
import torch.nn.functional as F

from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms

In [2]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'Data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf100_test_data = datasets.CIFAR100(root = 'Data', train = False, download= True, transform = transforms.ToTensor())

c:\Users\sanja\anaconda3fine\envs\torch_env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
# create validation set
print(len(cf10_training_data))


# print(len(cf100_training_data))

cf10_val_data = random_split(cf10_training_data, [int(len(cf10_training_data)*0.8), int(len(cf10_training_data)*0.2)])
#cf100_val_data = random_split(cf100_training_data, [len(cf100_training_data)*0.8, len(cf100_training_data)*0.2])

50000


In [4]:
print(1)
training_loader = torch.utils.data.DataLoader(cf10_training_data, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_data, batch_size=32, shuffle=True)
testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

1


In [27]:
class LesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = X.view(-1, 16*5*5)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)
        

In [28]:
torch.manual_seed(42)
print(1)
model_1 = LesNet()

1


In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_1.parameters(), lr=0.001)

In [33]:
def training_model(model, nr_epochs):
    losses = []
    for epoch in range(nr_epochs):
        running_loss = 0

        for i , data in enumerate(training_loader):
            inputs, label = data

            if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()

            else:
                model.cpu()

            optimizer.zero_grad()
            outputs = model.forward(inputs)
            loss = criterion(outputs, label)

            

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

           
        print(f"Epoch {epoch+1}, Loss : {running_loss/len(training_loader)}")


            
training_model(model_1, 20)

Epoch 1, Loss : 0.6369103288333994
Epoch 2, Loss : 0.6153277237962166
Epoch 3, Loss : 0.5929202791005468
Epoch 4, Loss : 0.579831842268726
Epoch 5, Loss : 0.5575050233803113
Epoch 6, Loss : 0.5401159366181625
Epoch 7, Loss : 0.5170144417006772
Epoch 8, Loss : 0.5014664024229013
Epoch 9, Loss : 0.4959988846841029
Epoch 10, Loss : 0.4690049048262915
Epoch 11, Loss : 0.46059947448019334
Epoch 12, Loss : 0.44959566102709364
Epoch 13, Loss : 0.43241963909268
Epoch 14, Loss : 0.42438795769102133
Epoch 15, Loss : 0.40865644006033547
Epoch 16, Loss : 0.3979248094085845
Epoch 17, Loss : 0.39133635349690876
Epoch 18, Loss : 0.37812054538366435
Epoch 19, Loss : 0.37174718776003945
Epoch 20, Loss : 0.360985985312966
